# 5 — Tabelas de Dados


| Seção | Conteúdo |
|------|----------|
| 5.1 | Importações e configurações |
| 5.2 | Carga dos dados tratados |
| 5.3 | Filtros utilizados |
| 5.4 | Agrupamentos disponíveis |
| 5.5 | Tabela selecionada |
| 5.6 | Geração de todas as tabelas |
| 5.7 | Exportação para CSV |
| 5.8 | Legenda dos indicadores |


#### 5.1 Importações e configurações

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    carregar_dados,
    calcular_indicadores,
)


2026-06-14 18:53:39.043 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


#### 5.2 Carga dos dados tratados

In [2]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print("Shape:", df_completo.shape)
df_completo.head(3)


Shape: (10285, 18)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino


#### 5.3 Filtros utilizados

In [3]:
anos_disponiveis = sorted(df_completo["Ano"].dropna().unique())
periodo = (int(min(anos_disponiveis)), int(max(anos_disponiveis)))
tipos_selecionados = sorted(df_completo["Tipo de Curso"].dropna().unique())

df = df_completo[
    (df_completo["Ano"] >= periodo[0])
    & (df_completo["Ano"] <= periodo[1])
    & (df_completo["Tipo de Curso"].isin(tipos_selecionados))
].copy()

print("Período:", periodo[0], "a", periodo[1])
print("Tipos de curso:", tipos_selecionados)
print("Matrículas distintas:", df["Código da Matricula"].nunique())
print("Linhas filtradas:", len(df))


Período: 2017 a 2024
Tipos de curso: ['Superior-Licenciatura', 'Superior-Tecnologia', 'Técnico-Integrado', 'Técnico-PROEJA - Integrado', 'Técnico-Subsequente']
Matrículas distintas: 3154
Linhas filtradas: 10285


#### 5.4 Agrupamentos disponíveis

In [4]:
agrupamento_opcoes = {
    "Por Ano (campus completo)": ["Ano"],
    "Por Ano e Curso": ["Ano", "Nome de Curso"],
    "Por Ano e Tipo de Curso": ["Ano", "Tipo de Curso"],
    "Por Ano e Eixo Tecnológico": ["Ano", "Eixo Tecnológico"],
    "Média por Curso (período)": ["Nome de Curso"],
    "Média por Tipo (período)": ["Tipo de Curso"],
}

pd.DataFrame(
    [{"Opção": nome, "Agrupamento": ", ".join(colunas)} for nome, colunas in agrupamento_opcoes.items()]
)


,Opção,Agrupamento
0,Por Ano (campus completo),Ano
1,Por Ano e Curso,"Ano, Nome de Curso"
2,Por Ano e Tipo de Curso,"Ano, Tipo de Curso"
3,Por Ano e Eixo Tecnológico,"Ano, Eixo Tecnológico"
4,Média por Curso (período),Nome de Curso
5,Média por Tipo (período),Tipo de Curso


#### 5.5 Tabela selecionada

In [5]:
colunas_identificacao = ["Ano", "Nome de Curso", "Tipo de Curso", "Eixo Tecnológico"]
colunas_indicadores = ["TC", "TE", "TR", "IEf", "TPE"]
colunas_contagens = [
    "matr_atendidas",
    "ingressantes",
    "concluintes",
    "evadidos",
    "MREG",
    "MRET",
    "conc_no_prazo",
    "conc_com_atraso",
    "abandono",
    "desligados",
    "transf_ext",
    "transf_int",
]


def montar_tabela_indicadores(df_base, agrupamento_escolhido):
    agrupamento = agrupamento_opcoes[agrupamento_escolhido]
    df_tabela = calcular_indicadores(df_base, agrupamento)
    colunas_exibir = [
        coluna for coluna in (colunas_identificacao + colunas_indicadores + colunas_contagens)
        if coluna in df_tabela.columns
    ]
    df_exibir = df_tabela[colunas_exibir].copy()
    colunas_grad = [coluna for coluna in colunas_indicadores if coluna in df_exibir.columns]
    return df_exibir, colunas_grad


agrupamento_escolhido = "Por Ano (campus completo)"
df_exibir, colunas_grad = montar_tabela_indicadores(df, agrupamento_escolhido)

print("Resultado:", agrupamento_escolhido)
print(f"{len(df_exibir)} linhas · {len(df_exibir.columns)} colunas")

df_exibir.style.format(
    {coluna: "{:.1f}" for coluna in colunas_grad}
).background_gradient(
    subset=colunas_grad,
    cmap="Blues",
)


Resultado: Por Ano (campus completo)
8 linhas · 18 colunas


,Ano,TC,TE,TR,IEf,TPE,matr_atendidas,ingressantes,concluintes,evadidos,MREG,MRET,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int
0,2017,8.4,15.8,5.2,34.3,78.5,811,292,70,130,569,42,48,20,70,45,13,2
1,2018,7.8,9.8,7.7,44.0,82.4,1026,344,80,102,765,79,67,13,40,46,15,1
2,2019,6.1,11.1,5.7,35.5,83.2,1285,373,79,142,991,73,61,17,103,33,6,0
3,2020,0.2,7.5,13.6,2.1,78.7,1223,218,2,95,960,166,0,2,78,10,4,3
4,2021,2.5,5.7,23.9,30.4,70.4,1122,147,28,64,762,268,0,28,27,18,19,0
5,2022,7.7,9.5,35.4,44.6,55.1,1464,290,112,139,695,518,50,62,83,36,20,0
6,2023,10.1,5.3,33.1,65.5,61.5,1609,389,163,86,827,533,97,66,47,21,18,0
7,2024,5.6,2.4,35.4,70.3,62.2,1745,358,99,41,988,617,47,50,1,32,8,0


#### 5.6 Geração de todas as tabelas

In [6]:
tabelas = {}
resumo_tabelas = []

for nome in agrupamento_opcoes:
    tabela, colunas_grad_tabela = montar_tabela_indicadores(df, nome)
    tabelas[nome] = tabela
    resumo_tabelas.append(
        {
            "Agrupamento": nome,
            "Linhas": len(tabela),
            "Colunas": len(tabela.columns),
            "Indicadores": ", ".join(colunas_grad_tabela),
        }
    )

pd.DataFrame(resumo_tabelas)


,Agrupamento,Linhas,Colunas,Indicadores
0,Por Ano (campus completo),8,18,"TC, TE, TR, IEf, TPE"
1,Por Ano e Curso,86,19,"TC, TE, TR, IEf, TPE"
2,Por Ano e Tipo de Curso,40,19,"TC, TE, TR, IEf, TPE"
3,Por Ano e Eixo Tecnológico,47,19,"TC, TE, TR, IEf, TPE"
4,Média por Curso (período),11,18,"TC, TE, TR, IEf, TPE"
5,Média por Tipo (período),5,18,"TC, TE, TR, IEf, TPE"


#### 5.7 Exportação para CSV

In [7]:
"""
def normalizar_nome_arquivo(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = texto.lower().replace(" ", "_").replace("/", "_")
    for caractere in ["(", ")", "—", "-", ":"]:
        texto = texto.replace(caractere, "_")
    while "__" in texto:
        texto = texto.replace("__", "_")
    return texto.strip("_")


pasta_saida = "../resultados"
os.makedirs(pasta_saida, exist_ok=True)
arquivos_exportados = []

for nome, tabela in tabelas.items():
    nome_arquivo = f"indicadores_{normalizar_nome_arquivo(nome)}.csv"
    caminho = os.path.join(pasta_saida, nome_arquivo)
    tabela.to_csv(caminho, index=False, sep=";", decimal=",")
    arquivos_exportados.append({"Agrupamento": nome, "Arquivo": caminho})

pd.DataFrame(arquivos_exportados)
"""

'\ndef normalizar_nome_arquivo(texto):\n    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")\n    texto = texto.lower().replace(" ", "_").replace("/", "_")\n    for caractere in ["(", ")", "—", "-", ":"]:\n        texto = texto.replace(caractere, "_")\n    while "__" in texto:\n        texto = texto.replace("__", "_")\n    return texto.strip("_")\n\n\npasta_saida = "../resultados"\nos.makedirs(pasta_saida, exist_ok=True)\narquivos_exportados = []\n\nfor nome, tabela in tabelas.items():\n    nome_arquivo = f"indicadores_{normalizar_nome_arquivo(nome)}.csv"\n    caminho = os.path.join(pasta_saida, nome_arquivo)\n    tabela.to_csv(caminho, index=False, sep=";", decimal=",")\n    arquivos_exportados.append({"Agrupamento": nome, "Arquivo": caminho})\n\npd.DataFrame(arquivos_exportados)\n'

#### 5.8 Legenda dos indicadores

| Sigla | Significado |
|-------|-------------|
| TC | Taxa de Conclusão (%) |
| TE | Taxa de Evasão (%) |
| TR | Taxa de Retenção (%) |
| IEf | Índice de Eficiência (%) |
| TPE | Taxa de Permanência e Êxito (%) |
| MREG | Matrículas Ativas Regulares (contagem) |
| MRET | Matrículas Ativas Retidas (contagem) |
| matr_atendidas | Total de matrículas atendidas no período |
